### Install Dependecies

In [1]:
# ! pip install video-kf
# !pip install ftfy regex tqdm
# !pip install git+https://github.com/openai/CLIP.git
# !pip install opencv-python-headless
# !pip install einops
# !pip install transformers
# !pip install -U sentence-transformers
# !pip install umap-learn
# !pip install tf-keras
# !pip install scipy
# !pip install seaborn
# !pip install ffmpeg
# !pip install keybert
# ! pip install gensim
# !pip install nltk

### import dependencies

In [1]:
import torch, torchvision
import numpy as np
import pandas as pd
import sklearn
import random
import os
import csv
import cv2
import glob
import regex
import einops
import umap
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
from tqdm import tqdm
from keybert import KeyBERT
# from tqdm.auto import tqdm
# from tqdm import tqdm
from torchvision.transforms import v2
from sklearn.feature_extraction.text import TfidfVectorizer

from collections import Counter
from ast import literal_eval
from transformers import pipeline
import videokf as vf


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### directories to save pipeline results

In [2]:
# vid_keyframes_dir = 'vid_keyframes'
vid_keyframes_dir = "vid_keyframes_"
trecvid_videos = 'trecvec_videos'
trecvid_files = os.listdir(trecvid_videos)
# processed_files = 'processed_files'
processed_files = 'processed_files_'
renamed_trecvid_videos  = 'renamed_trecvid_videos'
rename_vid_files = os.listdir(renamed_trecvid_videos)

print(os.path.isdir(vid_keyframes_dir), os.path.isdir(trecvid_videos)),
print(os.path.isdir(renamed_trecvid_videos))


True True
True


In [3]:
print(len(trecvid_files))

9000


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
print(device)

cpu


In [6]:
# import clip, a multimodal model for image embedding extraction
import clip
model, preprocess = clip.load('ViT-B/32', device)

In [7]:
# for image description
from transformers import pipeline
captioner = pipeline("image-to-text", model="Salesforce/blip-image-captioning-large")

#### Extract keyframes from video file

In [45]:
class ExtractKeyFrames:
    """
        Extracts keyframes from a video file
    """
    def __init__(self, trecvid_vids_dir, vid_keyframes_dir, verbose=None):
        self.parent_path = trecvid_vids_dir
        self.vid_keyframes_dir = vid_keyframes_dir
        self.possible_black = []
        self.possible_white = []
        self.verbose = verbose
      

    def check_for_black_or_white_bg(self, vid_keyframes_dir) -> None:
        
        """
          Deletes all white and all black keyframe from a keyframe direcotry. A keyframe dir must have at least two keyframes
          to perform deletion.
          
        """
        for root, dirs, files in os.walk(vid_keyframes_dir, topdown=False):
          
            for file in files:
                
                # print(len(files))
                # read image
                full_path = os.path.join(root, file)
                full_dir_path = os.path.dirname(full_path)
                len_dir = len(os.listdir(full_dir_path))
                im = Image.open(full_path)
        
                if np.mean(im) <15:
                    
                    # check length of parent_dir
                    if len_dir > 2:
                        
                        # print("len_dir", len_dir)
                        self.possible_black.append(full_path)
                        os.remove(full_path)
                elif np.mean(im) > 242:
                    
                    # print("greater",os.path.dirname(full_path))
                    if len_dir > 2:
                        
                        # print("len_dir", len_dir)
                        self.possible_white.append(full_path)
                        os.remove(full_path)
        if self.verbose:
            print(f"found a total of {len(self.possible_black)} all black images and {len(self.possible_white)} all white images")  

    def extract_key_frames(self):
        
        """
          Extracts keyframes from a video file,
          creates a folder named vid_keyframes
          and saves these keyframes in them.
        """
        all_vid_files = os.listdir(self.parent_path)[:100]
        # print(len(all_vid_files))
        # print(int(all_vid_files[0].split('.')[0]))
        all_vid_files = sorted(all_vid_files, key=lambda x: int(x.split('.')[0].strip()))
        for idx, vid in tqdm(enumerate(all_vid_files)):
            # print(idx, vid)
            full_vid_path = os.path.join(self.parent_path, vid)
            file_name = os.path.splitext(full_vid_path)[0]
            file_name = file_name.split('/')[-1]
            try:
                os.mkdir(f"{self.vid_keyframes_dir}/{file_name}")
                new_dir = os.path.join(self.vid_keyframes_dir, file_name)
            except FileExistsError:
                new_dir = os.path.join(self.vid_keyframes_dir, file_name)
                # if len(os.listdir(new_dir))== 0:
                #   # print("yesss", new_dir)
            vf.extract_keyframes(full_vid_path, output_dir_keyframes=new_dir)

    def __call__(self):
        # self.extract_key_frames()
        self.check_for_black_or_white_bg(self.vid_keyframes_dir)

#### process extracted keyframes to embeddings

In [46]:
class ProcessKeyFrameToEmbeddings:
    """
        Extract frame embeddings from key frames
    """
    def __init__(self, root_dir, verbose=None):
        self.root_dir = root_dir
        self.count = 1
        self.keyframe_embeddings = []
        self.verbose = verbose

    def get_frame_from_dir(self, root_dir):
        """
            Loops through provided directory of key frames, extracts embeddings from
            each key frame and saves the filename and associated embeddings as a dictionary
    
            args: root_dir -> A path to a directory of keyframes belonging to a specific video
            returns: A list of dictionaries of keyframes and filename
        """
        for idx, dir in enumerate(sorted(os.listdir(root_dir), key=lambda x: int(x.split('.')[0]))):
            full_dir_path = os.path.join(root_dir, dir)
            cur_vid_embeddings = []
            for file in os.listdir(full_dir_path):
                full_file_path = os.path.join(full_dir_path, file)
                img = Image.open(full_file_path)
                processed_img = preprocess(img).unsqueeze(0).to(device)
                with torch.no_grad():
                  embeddings = model.encode_image(processed_img)
                  cur_vid_embeddings.append(embeddings)
              # if len(cur_vid_embeddings) > 1:
              # print(len(cur_vid_embeddings))
            stacked_embeddings = torch.stack(cur_vid_embeddings, axis=0)
            average_embeddings = torch.mean(stacked_embeddings, axis=0).numpy(force=True)
            average_embeddings = np.squeeze(average_embeddings).tolist()
            cur_vid_embeddings = average_embeddings
            cur_load = {}
            cur_load['file_name'] = dir
            cur_load['embeddings'] = cur_vid_embeddings
            self.keyframe_embeddings.append(cur_load)
          # print(len(cur_vid_embeddings), cur_vid_embeddings[0].shape)

        return self.keyframe_embeddings



    def __call__(self):
        return self.get_frame_from_dir(self.root_dir)


In [47]:
def save_processed_embedding(dir_to_save, keyframe_embeddings):
    field_names = ['file_name', 'embeddings']   
    with open(f'{dir_to_save}/keyframes_embeddings.csv', 'w') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=field_names)
        writer.writeheader()
        writer.writerows(keyframe_embeddings)

### Cluster Embeddings using KMEANS

#### project embedding to lower dimension

In [48]:
class ReadProjectKeyFrameDF:
    """
        Project embeddings to lower dimension for improved clustering
    """
    def __init__(self, keyframe_csv, n_components, n_neighbours, random_state=42):
        self.keyframe_csv = keyframe_csv
        self.n_components = n_components
        self.n_neighbours = n_neighbours
        self.random_state = random_state

    def read_csv(self, keyframe_csv):
        """
            Reads a CSV file of video filenames and embeddings
            Args: key_frame_csv  A CSV file containing video files names and associated embeddings
            Returns: A dataframe of embeddings arrays
        """
        dataframe = pd.read_csv(keyframe_csv, converters={'embeddings': literal_eval})
        return dataframe

    def explode_df(self, dataframe):
        """
            Explodes the embedings rows as single column per embeding value
            Args: dataframe a dataframe of filename and associated embeddings
            Returns: A dataframe of embeddings of shape (num_videos, num_features)
        """
        exploded_df = pd.DataFrame(dataframe['embeddings'].to_list(), columns=[f'{i}' for i in range(len(dataframe['embeddings'].max()))])
        return exploded_df

    def project_to_lower_dim(self, dataframe, exploded_df):
        """
            Projects Embeddings to lower dimension
            Args: dataframe a dataframe filenames and associated embedding array
                  exploded_df a dataframe of embeddings of shape (num_videos, num_features)
        """
        #instantiate reducer object
        # dataframe = self.read_csv(self.keyframe_csv)
        # exploded_df = self.explode_df(dataframe)
        reducer = umap.UMAP(random_state = self.random_state,
                        n_components=self.n_components,
                        n_neighbors=self.n_neighbours)
        vid_embeddings_reduced = reducer.fit_transform(exploded_df).tolist()
        # make a copy of the original df and append the new embeddings
        reduced_df = dataframe.copy()
        reduced_df = reduced_df.drop('embeddings', axis=1)
        reduced_df['embeddings'] = vid_embeddings_reduced
    
        return reduced_df

    def __call__(self):
        dataframe = self.read_csv(self.keyframe_csv)
        exploded_df = self.explode_df(dataframe)
        reduced_df = self.project_to_lower_dim(dataframe, exploded_df)
    
        return reduced_df

In [49]:
class ClusterEmbeddings:
    def __init__(self, vid_frame_embeddings, processed_files_dir, k, min_cluster_size, recluster_k, isolate_large=False) -> None:
        self.vid_frame_embeddings = vid_frame_embeddings
        self.processed_files_dir = processed_files_dir
        self.k = k
        self.min_cluster_size = min_cluster_size
        self.recluster_k = recluster_k
        self.isolate_large = isolate_large
        self.start = False
        self.large_clusters = []
        self.final_cluster = pd.DataFrame()

    def explode_embeddings(self, dataframe):
        print("exploding embeddings")
        exploded_df = pd.DataFrame(dataframe['embeddings'].to_list(),
                                   columns=[f"{i}" for i in range(len(dataframe['embeddings'].max()))])
        print("explode_rem done...")
        return dataframe, exploded_df
    
        # exploded_df = pd.DataFrame(dataframe['embeddings'].to_list(),
        #                            columns=[f'{i}' for i in range(len(dataframe['embeddings'].max()))])
        # return exploded_df, dataframe

    def check_large_clusters(self, unique_cluster_ids, dataframe):
        cur_final_clusters = pd.DataFrame()
        for idx in unique_cluster_ids:
          assoc_cluster = dataframe[dataframe['cluster_id']==idx]
          if len(assoc_cluster.index) > self.min_cluster_size:
            self.large_clusters.append(assoc_cluster)
          else:
            cur_final_clusters = pd.concat([cur_final_clusters, assoc_cluster], ignore_index=True)
    
        self.final_cluster = pd.concat([self.final_cluster, cur_final_clusters], ignore_index=True)
        return self.final_cluster


    def cluster_df(self, dataframe, exploded_df):
        if not self.start:
          kmeans = KMeans(n_clusters=self.k, random_state=42)
          cluster_ids = kmeans.fit_predict(exploded_df)
          unique_cluster_ids = np.unique(cluster_ids)
          dataframe['cluster_id'] = cluster_ids
          if not self.isolate_large:
            return dataframe
          else:
            final_cluster = self.check_large_clusters(unique_cluster_ids, dataframe)
          return final_cluster
        else:
          kmeans = KMeans(n_clusters=self.recluster_k, random_state=42)
          cluster_ids = kmeans.fit_predict(exploded_df)
          dataframe['cluster_id'] = cluster_ids
          return dataframe

    def write_to_csv(self, save_to_file=None, clustered_df=None):
        # read final clusters df
        final_cluster_without_embs = self.final_cluster.drop('embeddings', axis=1)
        # clustered_videos_csv = f"{BASE_TRECVID25_DIR}/vid_clusters.csv"
        final_cluster_without_embs.to_csv(f"{self.processed_files_dir}/vid_clusters.csv", index=False, mode='w')
        return f"{self.processed_files_dir}/vid_clusters.csv"


    def __call__(self):
        if not self.isolate_large:
          dataframe, exploded_df = self.explode_embeddings(self.vid_frame_embeddings)
          self.final_cluster = self.cluster_df(dataframe, exploded_df)
        else:
          if not self.start:
            dataframe, exploded_df = self.explode_embeddings(self.vid_frame_embeddings)
            self.final_cluster = self.cluster_df(dataframe, exploded_df)
    
          while self.large_clusters:
            print("entered while", len(self.large_clusters))
            cur_df = self.large_clusters.pop()
            self.final_cluster = pd.concat([self.final_cluster, cur_df], ignore_index=True)
    
            # dataframe, exploded_df = self.explode_embeddings(cur_df)
            # cur_df_clustered = self.cluster_df(dataframe, exploded_df)
            # self.final_cluster = pd.concat([self.final_cluster, cur_df_clustered], ignore_index=True)
    
        # write self.final_cluster w/o embedding column to csv
        vid_clusters_csv_file_path = self.write_to_csv(save_to_file=self.processed_files_dir, clustered_df=self.final_cluster)
        return self.final_cluster, vid_clusters_csv_file_path

In [140]:
class DescribeKeyFramesAndExtractKeywords:
    """
        Described the activities in a frame and extracts keywords from
        the description.
    """
    def __init__(self, vid_keyframe_dir, processed_files, keyword_model, vid_clusters_csv_file_path):
        self.vid_keyframe_dir = vid_keyframe_dir
        self.processed_files = processed_files
        self.keyword_model = keyword_model
        self.clustered_csv = vid_clusters_csv_file_path

    def describe_frames(self):
        """
            Loops through keyframes belonging to a video and concatenates them as strings
            returns: the filepath to the csv of filename and associated keyframes
        """
        fields = ['filename', 'caption']
        with open(f"{self.processed_files}/caption_csv", 'w') as csv_file:
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow(fields)
        all_captions = []
        keyframe_dirs = os.listdir(self.vid_keyframe_dir)
        # print("len_keyframe_dirs", keyframe_dirs)
        for idx, dir_ in enumerate(keyframe_dirs):
            full_dir_path = os.path.join(self.vid_keyframe_dir, dir_)
            dir_name = os.path.basename(dir_)
            cur_caption = ""
            for img in os.listdir(full_dir_path):
              img_full_path = os.path.join(full_dir_path, img)
              # read img
              pil_image = Image.open(img_full_path)
              keyframe_caption = captioner(pil_image)
              keyframe_caption = keyframe_caption[0]['generated_text']
        
              cur_caption += keyframe_caption + " "
              # print(img_full_path)
            cur_vid_name_cap = [dir_name, cur_caption]
            all_captions.append(cur_vid_name_cap)
            # print("all_captions", all_captions)
            # return "done"
        with open(f"{self.processed_files}/caption_csv", 'a', newline='') as file_c:
            csv_writer = csv.writer(file_c)
            csv_writer.writerows(all_captions)
        return f"{self.processed_files}/caption_csv"
    def extract_keywords(self):
        caption_csv = self.describe_frames()
        df = pd.read_csv(caption_csv)
        
        all_captions = df['caption'].tolist()
        all_key_words = self.keyword_model.extract_keywords(all_captions)
        
        keywords_load = []
        top_keywords = []
        for keywords in all_key_words:
            # print("keywords", keywords )
            keywords = list(filter(lambda x: x[0] not in ["arafed", "arafe", "araf"], keywords))
            idx = 0
            first_keyword = keywords[idx][0]
            # eliminate all appearances of arafed
            if (first_keyword == "arafed"):
                idx += 1
            first_keyword = keywords[idx][0] 
            idx +=1
            second_keyword = keywords[idx][0]
            idx +=1
            try:
                third_keyword = keywords[idx][0]
                cur_top_keyword = f"{first_keyword}_{second_keyword}_{third_keyword}"
            except:
                cur_top_keyword = f"{first_keyword}_{second_keyword}"
            top_keywords.append(cur_top_keyword)
            cur_keyword = ""
            for keyword in keywords:
                if keyword[0] == "arafe":
                    continue
                cur_keyword += keyword[0] + " "
            keywords_load.append(cur_keyword)
        
        return np.array(keywords_load), np.array(top_keywords), caption_csv, all_captions

    def assign_top_keywords_to_caption_csv(self):
        keywords_load, top_keywords, caption_csv, all_captions = self.extract_keywords()
        df = pd.read_csv(caption_csv)
        new_df = df.drop('caption', axis=1)
        new_caption = np.array(all_captions)
        new_df['captions'] = new_caption
        new_df['keywords'] = keywords_load
        new_df['top_keywords'] = top_keywords
        new_df = new_df.assign(cluster_ids="")
        # read initial clustered csv
        clustered_csv_df = pd.read_csv(self.clustered_csv)
        for idx, row in clustered_csv_df.iterrows():
            file_name = row['file_name']
            cluster_id = row['cluster_id']
            new_df.loc[new_df['filename'] == file_name, 'cluster_ids'] = cluster_id
        # save file
        new_df = new_df.assign(cluster_keywords="")    
        new_df.to_csv(f"{self.processed_files}/vids_clusters_keywords.csv", index=False, mode='w')

        return f"{self.processed_files}/vids_clusters_keywords.csv"

    def extract_keywords_per_cluster(self, vids_clusters_keywords_csv):
        cluster_keywords = {}
        df = pd.read_csv(vids_clusters_keywords_csv)
        unique_cluster_ids = df['cluster_ids'].unique()
    
        for c_id in unique_cluster_ids:
            assoc_cluster = df[df['cluster_ids'] == c_id]
            cur_keywords = ""
            for idx, row in assoc_cluster.iterrows():
                assoc_keywords = row['keywords']
                assoc_keywords = assoc_keywords.replace('arafy', "")
                assoc_keywords = assoc_keywords.replace('araffe', "")
                cur_keywords += assoc_keywords + " "  
    
            vectorizer = TfidfVectorizer(stop_words='english')
            tfidf_matrix = vectorizer.fit_transform([cur_keywords])
    
            feature_names = vectorizer.get_feature_names_out()
            tfidf_scores = tfidf_matrix.toarray()[0]
    
            word_tfidf = list(zip(feature_names, tfidf_scores))
    
            word_tfidf = sorted(word_tfidf, key=lambda x: x[1], reverse=True)[:4]
            word_tfidf = [word_score[0] for word_score in word_tfidf]
            word_tfidf = "_".join(word_tfidf)
            
            cluster_keywords[c_id] = word_tfidf
    
        return cluster_keywords, df
        

    def __call__(self):
        vids_clusters_keywords_csv = self.assign_top_keywords_to_caption_csv()
        cluster_keywords, df = self.extract_keywords_per_cluster(vids_clusters_keywords_csv)
        
        

        for idx, row in df.iterrows():
            cluster_id = row['cluster_ids']
            cluster_keyword = cluster_keywords[cluster_id]
            # print(f"cluster_id: {cluster_id} \n cluster_keyword: {type(cluster_keyword)}")
            # assoc_row = df.loc[df['cluster_ids']==cluster_id]
            df.loc[df['cluster_ids']==cluster_id, 'cluster_keywords'] = cluster_keyword
        # new_df.to_csv(f"{self.processed_files}/vids_clusters_keywords.csv", index=False, mode='w')
        df.to_csv(vids_clusters_keywords_csv, index=False, mode='w')
        print("done assigning cluster_keywords")

        return df
        
    

In [141]:
class RunPipeline:
    def __init__(self, trecvid_videos, vid_keyframes_dir, processed_files, keyword_model=None, n_components=25, n_neighbours=9, random_state=42, verbose=False) -> None:
        self.verbose = verbose
        self.trecvid_vids_dir = trecvid_videos
        self.vid_keyframes_dir = vid_keyframes_dir
        self.processed_files = processed_files
        self.n_components = n_components
        self.n_neighbours = n_neighbours
        self.random_state = random_state
        self.keyword_model = keyword_model
        
    def save_processed_embeddings(self, dir_to_save, keyframe_embeddings):
        field_names = ['file_name', 'embeddings']   
        with open(f'{dir_to_save}/keyframes_embeddings.csv', 'w') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=field_names)
            writer.writeheader()
            writer.writerows(keyframe_embeddings)
        return f'{dir_to_save}/keyframes_embeddings.csv'

    def __call__(self):
        # extract keyframes
        print("extracting keyframes")
        extract_keyframes = ExtractKeyFrames(self.trecvid_vids_dir, self.vid_keyframes_dir, verbose= self.verbose)
        extract_keyframes()
        print("Done extracting...")
        
        print("extracting embeddings from keyframes")
        process_keyframe_to_embeddings = ProcessKeyFrameToEmbeddings(self.vid_keyframes_dir, verbose=self.verbose)
        keyframe_embeddings = process_keyframe_to_embeddings()
        print("Done extracting embeddings from keyframes")

        print("saving extracted embeddings to file")
        saved_embeddings_csv_file = self.save_processed_embeddings(self.processed_files, keyframe_embeddings)
        print("Done saving extracted embeddings  to file")

        print("reading keyframe embeddings and projecting to lower dimensions")
        read_project_keyframe_df = ReadProjectKeyFrameDF(saved_embeddings_csv_file, n_components=25, n_neighbours=9, random_state=42)
        reduced_df = read_project_keyframe_df()
        print("Done reading keyframe embeddings and projecting to lower dimensions")

        print("clustering with Kmeans")
        cluster_embeddings = ClusterEmbeddings(vid_frame_embeddings=reduced_df, processed_files_dir= self.processed_files, k=50, min_cluster_size=300, recluster_k=10, isolate_large=False)
        clusters, vid_clusters_csv_file_path = cluster_embeddings()
        print("Done clustering with Kmeans")

        print("describing keyframes and extracting keywords")
        describe_keyframes_and_extractkeywords = DescribeKeyFramesAndExtractKeywords(self.vid_keyframes_dir, self.processed_files, self.keyword_model, vid_clusters_csv_file_path)
        return describe_keyframes_and_extractkeywords()
        print("done, processing file")
        

In [142]:
keyword_model = KeyBERT()
run_pipeline = RunPipeline(trecvid_videos, vid_keyframes_dir, processed_files, keyword_model=keyword_model, n_components=25, n_neighbours=9, random_state=42, verbose=False)

In [143]:
df = run_pipeline()

extracting keyframes
Done extracting...
extracting embeddings from keyframes
Done extracting embeddings from keyframes
saving extracted embeddings to file
Done saving extracted embeddings  to file
reading keyframe embeddings and projecting to lower dimensions


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch/lib/python3.11/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Done reading keyframe embeddings and projecting to lower dimensions
clustering with Kmeans
exploding embeddings
explode_rem done...
Done clustering with Kmeans
describing keyframes and extracting keywords


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch/lib/python3.11/site-packages/transformers/generation/utils.py:1141: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


done assigning cluster_keywords


In [144]:
df.head(10)

,filename,captions,keywords,top_keywords,cluster_ids,cluster_keywords
0,10866,a close up of a poster on a wall with a microp...,poster microphone wall building reads,poster_microphone_wall,17,microphone_banner_boat_building
1,11518,there is a man that is standing in the dark wi...,poster standing aviation dark phone,poster_standing_aviation,2,phone_aviation_camera_cell
2,14258,woman in a red and black dress is dancing with...,dancing woman game people playing,dancing_woman_game,9,dancing_woman_arafish_bathing
3,10098,arafed view of a house in a wooded area with a...,sunset forest wooded view parked,sunset_forest_wooded,37,parked_building_dirt_door
4,11723,arafed man sitting in a chair with a tie on ar...,chair sitting phone talking tie,chair_sitting_phone,18,sitting_talking_camera_chair
5,12334,there is a man that is standing in the dark wi...,dark phone standing cell man,dark_phone_standing,49,cell_dark_man_phone
6,17474,araffe on stage with a microphone and a microp...,microphone stage guitar stand araffe,microphone_stage_guitar,25,background_cigarette_guitar_image
7,9895,arafed man with a hat and a white shirt is tal...,microphone recording studio hat talking,microphone_recording_studio,41,glass_hat_microphone_recording
8,9250,there is a man that is holding a microphone in...,microphone remote hand red light,microphone_remote_hand,29,bench_hand_light_microphone
9,13876,people in red outfits are dancing in a parade ...,parade dancing lanterns outfits dresses,parade_dancing_lanterns,9,dancing_woman_arafish_bathing


In [145]:
csv_path = 'processed_files_/vids_clusters_keywords.csv'

In [146]:
df = pd.read_csv(csv_path)

In [147]:
df.head(10)

,filename,captions,keywords,top_keywords,cluster_ids,cluster_keywords
0,10866,a close up of a poster on a wall with a microp...,poster microphone wall building reads,poster_microphone_wall,17,microphone_banner_boat_building
1,11518,there is a man that is standing in the dark wi...,poster standing aviation dark phone,poster_standing_aviation,2,phone_aviation_camera_cell
2,14258,woman in a red and black dress is dancing with...,dancing woman game people playing,dancing_woman_game,9,dancing_woman_arafish_bathing
3,10098,arafed view of a house in a wooded area with a...,sunset forest wooded view parked,sunset_forest_wooded,37,parked_building_dirt_door
4,11723,arafed man sitting in a chair with a tie on ar...,chair sitting phone talking tie,chair_sitting_phone,18,sitting_talking_camera_chair
5,12334,there is a man that is standing in the dark wi...,dark phone standing cell man,dark_phone_standing,49,cell_dark_man_phone
6,17474,araffe on stage with a microphone and a microp...,microphone stage guitar stand araffe,microphone_stage_guitar,25,background_cigarette_guitar_image
7,9895,arafed man with a hat and a white shirt is tal...,microphone recording studio hat talking,microphone_recording_studio,41,glass_hat_microphone_recording
8,9250,there is a man that is holding a microphone in...,microphone remote hand red light,microphone_remote_hand,29,bench_hand_light_microphone
9,13876,people in red outfits are dancing in a parade ...,parade dancing lanterns outfits dresses,parade_dancing_lanterns,9,dancing_woman_arafish_bathing
